In [3]:
# ============================================================
# 8.1 + 9. LEAKAGE-SAFE DATA + LINEAR REGRESSION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 8.1 LOAD DATA
# ============================================================

SCORED_PATH = r"C:/Users/muskan/OneDrive/Desktop/effisense/data/processed/mouse_productivity_scores.csv"

df = pd.read_csv(SCORED_PATH)

TARGET = "productivity_score"

BEHAVIORAL_FEATURES = [
    "mean_velocity",
    "std_velocity",
    "median_velocity",
    "max_velocity",
    "min_velocity",
    "mean_acceleration",
    "std_acceleration",
    "max_acceleration",
    "mean_direction_change",
    "median_direction_change",
    "total_distance",
    "mean_distance",
    "window_duration",
    "idle_events",
    "idle_time",
    "idle_percentage",
    "pause_count",
    "mean_pause",
    "median_pause",
    "std_pause",
    "pause_95",
    "velocity_entropy",
    "direction_entropy",
    "velocity_cv",
    "smoothness",
    "num_events"
]

X = df[BEHAVIORAL_FEATURES].copy()
y = df[TARGET].copy()


# ============================================================
# 8.2 CLEAN FEATURES
# ============================================================

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())


# ============================================================
# 8.3 SESSION-BASED SPLIT
# ============================================================

groups = (
    df["user"].astype(str)
    + "_"
    + df["session"].astype(str)
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

overlap = set(groups_train).intersection(set(groups_test))

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Train sessions:", groups_train.nunique())
print("Test sessions :", groups_test.nunique())
print("Session overlap:", len(overlap))


# ============================================================
# 9. LINEAR REGRESSION
# ============================================================

linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

# Keep predictions within BPI's 0–100 range
y_pred = np.clip(y_pred, 0, 100)


# ============================================================
# 9.1 EVALUATION
# ============================================================

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n==============================")
print("LINEAR REGRESSION RESULTS")
print("==============================")

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


# ============================================================
# 9.2 SAMPLE PREDICTIONS
# ============================================================

comparison = pd.DataFrame({
    "actual_score": y_test.values,
    "predicted_score": y_pred
})

print("\nSample predictions:")
print(comparison.head(10))

X_train: (7980, 26)
X_test : (2020, 26)
Train sessions: 72
Test sessions : 19
Session overlap: 0

LINEAR REGRESSION RESULTS
MAE  : 1.5496
MSE  : 5.1522
RMSE : 2.2698
R²   : 0.9934

Sample predictions:
   actual_score  predicted_score
0     71.042882        70.426653
1     49.248414        42.226037
2     50.297183        44.928997
3     51.618271        47.305865
4     33.251916        30.886146
5     35.827960        32.481760
6     34.413828        32.590442
7     34.720532        34.513377
8     34.859078        34.822578
9     49.445956        49.522395


In [4]:
# ============================================================
# 9.3 LINEAR REGRESSION FEATURE COEFFICIENTS
# ============================================================

# Extract coefficients from the trained Linear Regression model
coefficients = linear_model.named_steps["regressor"].coef_

coef_df = pd.DataFrame({
    "feature": BEHAVIORAL_FEATURES,
    "coefficient": coefficients
})

# Absolute coefficient = strength of contribution
coef_df["absolute_coefficient"] = (
    coef_df["coefficient"].abs()
)

# Sort strongest effects first
coef_df = coef_df.sort_values(
    "absolute_coefficient",
    ascending=False
)

print("Feature coefficients:")
print(
    coef_df[
        ["feature", "coefficient", "absolute_coefficient"]
    ].to_string(index=False)
)

Feature coefficients:
                feature  coefficient  absolute_coefficient
        window_duration   -21.504504             21.504504
              idle_time    20.780589             20.780589
        idle_percentage   -10.370470             10.370470
             smoothness     6.511040              6.511040
            velocity_cv    -4.895720              4.895720
          mean_velocity    -3.669112              3.669112
           std_velocity     3.321137              3.321137
       velocity_entropy     2.062935              2.062935
         total_distance     1.890219              1.890219
       std_acceleration     1.255692              1.255692
           max_velocity     1.250859              1.250859
               pause_95    -1.172755              1.172755
            idle_events    -1.092423              1.092423
            pause_count    -1.092423              1.092423
      mean_acceleration     0.858177              0.858177
        median_velocity     0.7074

In [5]:
# ============================================================
# 9.4 FEATURE CORRELATION / MULTICOLLINEARITY CHECK
# ============================================================

# Correlation matrix of the behavioral features
correlation_matrix = X_train.corr()

# Find highly correlated feature pairs
high_correlations = []

for i in range(len(correlation_matrix.columns)):
    for j in range(i + 1, len(correlation_matrix.columns)):

        feature_1 = correlation_matrix.columns[i]
        feature_2 = correlation_matrix.columns[j]

        correlation = correlation_matrix.iloc[i, j]

        if abs(correlation) >= 0.80:
            high_correlations.append({
                "feature_1": feature_1,
                "feature_2": feature_2,
                "correlation": correlation
            })

high_corr_df = pd.DataFrame(high_correlations)

# Sort by strongest correlation
if not high_corr_df.empty:
    high_corr_df["absolute_correlation"] = (
        high_corr_df["correlation"].abs()
    )

    high_corr_df = high_corr_df.sort_values(
        "absolute_correlation",
        ascending=False
    )

    print("Highly correlated feature pairs (|r| >= 0.80):")
    print(
        high_corr_df[
            ["feature_1", "feature_2", "correlation"]
        ].to_string(index=False)
    )

else:
    print("No feature pairs have |correlation| >= 0.80.")

Highly correlated feature pairs (|r| >= 0.80):
            feature_1               feature_2  correlation
          idle_events             pause_count     1.000000
      window_duration               idle_time     0.999142
     std_acceleration        max_acceleration     0.989292
         std_velocity            max_velocity     0.984103
            std_pause                pause_95     0.978181
     velocity_entropy              smoothness     0.948929
           mean_pause                pause_95     0.944708
    mean_acceleration        std_acceleration    -0.943943
          velocity_cv              smoothness    -0.943761
     velocity_entropy             velocity_cv    -0.938718
        mean_velocity            std_velocity     0.929067
           mean_pause               std_pause     0.920859
    mean_acceleration        max_acceleration    -0.893514
      window_duration                pause_95     0.884900
            idle_time                pause_95     0.883419
        m

In [6]:
# ============================================================
# 9.5 UNDERSTAND HIGHLY CORRELATED FEATURES
# ============================================================

CHECK_COLUMNS = [
    "window_duration",
    "idle_time",
    "idle_percentage",
    "idle_events",
    "pause_count",
    "std_acceleration",
    "max_acceleration",
    "std_velocity",
    "max_velocity",
    "std_pause",
    "pause_95",
    "mean_pause",
    "velocity_entropy",
    "velocity_cv",
    "smoothness"
]

print("Feature summary:\n")

print(
    df[CHECK_COLUMNS]
    .describe()
    .T[
        ["mean", "std", "min", "50%", "max"]
    ]
)


# ------------------------------------------------------------
# Check exact relationships
# ------------------------------------------------------------

print("\n\nKey relationships:")

print(
    "\nidle_events == pause_count:",
    (df["idle_events"] == df["pause_count"]).mean()
)

print(
    "\nwindow_duration / idle_time:"
)

print(
    df[
        ["window_duration", "idle_time"]
    ].head(10)
)

print(
    "\nVelocity-related features:"
)

print(
    df[
        ["mean_velocity", "std_velocity",
         "max_velocity", "velocity_cv", "smoothness"]
    ].head(10)
)

Feature summary:

                         mean           std         min          50%  \
window_duration    131.530439    109.714405   13.120666    87.271287   
idle_time          103.290405    110.464936    0.000000    59.373087   
idle_percentage     64.883757     22.309186    0.000000    68.618427   
idle_events          8.822100      3.985027    0.000000     9.000000   
pause_count          8.822100      3.985027    0.000000     9.000000   
std_acceleration  1570.828712   2847.719627  180.040674   884.948303   
max_acceleration  7093.741969  16045.298077  428.648812  3261.228720   
std_velocity      1221.656779   1971.664425   86.837035   585.735151   
max_velocity      6053.324200  10140.554119  442.000000  2780.000000   
std_pause           11.775161     14.374141    0.000000     5.827673   
pause_95            28.855644     31.463938    0.000000    15.441422   
mean_pause          10.393016      9.186284    0.000000     6.868683   
velocity_entropy     2.752469      0.847634   

In [13]:
# ============================================================
# SECTION 9.6 — CREATE REDUCED PRODUCTIVITY DATASET
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Load the CORRECT scored dataset
# ------------------------------------------------------------

FEATURE_PATH = r"C:/Users/muskan/OneDrive/Desktop/effisense/data/processed/mouse_productivity_scores.csv"

REDUCED_PATH = r"C:/Users/muskan/OneDrive/Desktop/effisense/data/processed/mouse_productivity_scores_reduced.csv"

df = pd.read_csv(FEATURE_PATH)

print("Original dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 2. Reduced behavioral features
# ------------------------------------------------------------

REDUCED_FEATURES = [
    # Engagement
    "idle_percentage",
    "pause_count",
    "mean_pause",

    # Movement
    "mean_velocity",
    "total_distance",
    "mean_distance",

    # Consistency
    "velocity_cv",
    "velocity_entropy",
    "smoothness",

    # Acceleration
    "mean_acceleration",
    "std_acceleration",

    # Direction
    "mean_direction_change",
    "direction_entropy",

    # Other
    "num_events"
]

ID_COLUMNS = [
    "user",
    "session",
    "window_id"
]

TARGET = "productivity_score"


# ------------------------------------------------------------
# 3. Verify required columns
# ------------------------------------------------------------

required = ID_COLUMNS + REDUCED_FEATURES + [TARGET]

missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(f"Missing columns: {missing}")

print("\nAll required columns found.")


# ------------------------------------------------------------
# 4. Create reduced dataset
# ------------------------------------------------------------

reduced_df = df[required].copy()

print("\nReduced dataset shape:", reduced_df.shape)


# ------------------------------------------------------------
# 5. Check missing values
# ------------------------------------------------------------

print("\nMissing values:")
print(reduced_df.isna().sum())


# ------------------------------------------------------------
# 6. Save reduced dataset
# ------------------------------------------------------------

reduced_df.to_csv(
    REDUCED_PATH,
    index=False
)

print("\nReduced dataset saved:")
print(REDUCED_PATH)


# ------------------------------------------------------------
# 7. Verification
# ------------------------------------------------------------

print("\nFinal shape:", reduced_df.shape)

print("Users:", reduced_df["user"].nunique())
print("Sessions:", reduced_df["session"].nunique())

print("\nProductivity score:")
print(reduced_df["productivity_score"].describe())

print("\nFirst 5 rows:")
print(reduced_df.head())

Original dataset shape: (10000, 49)

Columns:
['user', 'session', 'window_id', 'mean_velocity', 'std_velocity', 'median_velocity', 'max_velocity', 'min_velocity', 'mean_acceleration', 'std_acceleration', 'max_acceleration', 'mean_direction_change', 'median_direction_change', 'total_distance', 'mean_distance', 'window_duration', 'idle_events', 'idle_time', 'idle_percentage', 'pause_count', 'mean_pause', 'median_pause', 'std_pause', 'pause_95', 'velocity_entropy', 'direction_entropy', 'velocity_cv', 'smoothness', 'num_events', 'idle_percentage_norm', 'pause_count_norm', 'mean_pause_norm', 'mean_velocity_norm', 'mean_distance_norm', 'total_distance_norm', 'velocity_cv_norm', 'smoothness_norm', 'idle_score', 'pause_score', 'pause_duration_score', 'movement_rate', 'movement_rate_norm', 'velocity_stability_score', 'smoothness_score', 'efficiency_score', 'velocity_consistency_score', 'engagement_score', 'consistency_score', 'productivity_score']

All required columns found.

Reduced dataset s

In [14]:
# ============================================================
# CHECK CURRENT SESSION DISTRIBUTION
# ============================================================

print("Total sessions:", reduced_df["session"].nunique())

print("\nSessions per user:")
print(
    reduced_df.groupby("user")["session"]
    .nunique()
    .sort_values(ascending=False)
)

print("\nWindows per session:")
print(
    reduced_df.groupby(["user", "session"])
    .size()
    .describe()
)

Total sessions: 53

Sessions per user:
user
1001    52
1       11
1012     9
1006     8
1050     4
1048     3
1019     2
1010     1
1030     1
Name: session, dtype: int64

Windows per session:
count     91.000000
mean     109.890110
std      144.130069
min        1.000000
25%       22.500000
50%       54.000000
75%      158.000000
max      754.000000
dtype: float64


In [15]:
session_ids = df[['user', 'session']].drop_duplicates()

print("Total sessions:", len(session_ids))

print(
    df.groupby(['user', 'session'])
      .size()
      .describe()
)

Total sessions: 91
count     91.000000
mean     109.890110
std      144.130069
min        1.000000
25%       22.500000
50%       54.000000
75%      158.000000
max      754.000000
dtype: float64


In [16]:
# ============================================================
# 8.1 + 9. LEAKAGE-SAFE DATA + LINEAR REGRESSION
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# 8.1 LOAD DATA
# ============================================================

SCORED_PATH = r"C:/Users/muskan/OneDrive/Desktop/effisense/data/processed/mouse_productivity_scores_reduced.csv"

df = pd.read_csv(SCORED_PATH)

TARGET = "productivity_score"

BEHAVIORAL_FEATURES = [
# Engagement
    "idle_percentage",
    "pause_count",
    "mean_pause",

    # Movement
    "mean_velocity",
    "total_distance",
    "mean_distance",

    # Consistency
    "velocity_cv",
    "velocity_entropy",
    "smoothness",

    # Acceleration
    "mean_acceleration",
    "std_acceleration",

    # Direction
    "mean_direction_change",
    "direction_entropy",

    # Other
    "num_events"
]

X = df[BEHAVIORAL_FEATURES].copy()
y = df[TARGET].copy()


# ============================================================
# 8.2 CLEAN FEATURES
# ============================================================

X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median())


# ============================================================
# 8.3 SESSION-BASED SPLIT
# ============================================================

groups = (
    df["user"].astype(str)
    + "_"
    + df["session"].astype(str)
)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

overlap = set(groups_train).intersection(set(groups_test))

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Train sessions:", groups_train.nunique())
print("Test sessions :", groups_test.nunique())
print("Session overlap:", len(overlap))


# ============================================================
# 9. LINEAR REGRESSION
# ============================================================

linear_model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", LinearRegression())
])

linear_model.fit(X_train, y_train)

y_pred = linear_model.predict(X_test)

# Keep predictions within BPI's 0–100 range
y_pred = np.clip(y_pred, 0, 100)


# ============================================================
# 9.1 EVALUATION
# ============================================================

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n==============================")
print("LINEAR REGRESSION RESULTS")
print("==============================")

print(f"MAE  : {mae:.4f}")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"R²   : {r2:.4f}")


# ============================================================
# 9.2 SAMPLE PREDICTIONS
# ============================================================

comparison = pd.DataFrame({
    "actual_score": y_test.values,
    "predicted_score": y_pred
})

print("\nSample predictions:")
print(comparison.head(10))

X_train: (7980, 14)
X_test : (2020, 14)
Train sessions: 72
Test sessions : 19
Session overlap: 0

LINEAR REGRESSION RESULTS
MAE  : 1.7621
MSE  : 5.6938
RMSE : 2.3862
R²   : 0.9927

Sample predictions:
   actual_score  predicted_score
0     71.042882        69.890173
1     49.248414        40.756166
2     50.297183        44.578887
3     51.618271        46.930875
4     33.251916        32.119337
5     35.827960        32.631131
6     34.413828        33.585837
7     34.720532        36.213345
8     34.859078        36.836926
9     49.445956        51.011371


In [18]:
# ============================================================
# 9.3 RAW vs CLIPPED PREDICTION EVALUATION
# ============================================================

raw_pred = linear_model.predict(X_test)
clipped_pred = np.clip(raw_pred, 0, 100)

print("==============================")
print("RAW vs CLIPPED PREDICTIONS")
print("==============================")

print("Raw predictions below 0 :", (raw_pred < 0).sum())
print("Raw predictions above 100:", (raw_pred > 100).sum())

print("\n--- RAW PREDICTIONS ---")
print(f"MAE  : {mean_absolute_error(y_test, raw_pred):.4f}")
print(f"MSE  : {mean_squared_error(y_test, raw_pred):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, raw_pred)):.4f}")
print(f"R²   : {r2_score(y_test, raw_pred):.4f}")

print("\n--- CLIPPED PREDICTIONS ---")
print(f"MAE  : {mean_absolute_error(y_test, clipped_pred):.4f}")
print(f"MSE  : {mean_squared_error(y_test, clipped_pred):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, clipped_pred)):.4f}")
print(f"R²   : {r2_score(y_test, clipped_pred):.4f}")

RAW vs CLIPPED PREDICTIONS
Raw predictions below 0 : 52
Raw predictions above 100: 105

--- RAW PREDICTIONS ---
MAE  : 2.2552
MSE  : 11.6892
RMSE : 3.4189
R²   : 0.9850

--- CLIPPED PREDICTIONS ---
MAE  : 1.7621
MSE  : 5.6938
RMSE : 2.3862
R²   : 0.9927


In [19]:
# ============================================================
# 9.4 FEATURE COEFFICIENT ANALYSIS
# ============================================================

# Get trained Linear Regression coefficients
regressor = linear_model.named_steps["regressor"]

coefficients = pd.DataFrame({
    "feature": BEHAVIORAL_FEATURES,
    "coefficient": regressor.coef_
})

# Absolute coefficient = strength of contribution
coefficients["abs_coefficient"] = (
    coefficients["coefficient"].abs()
)

# Sort by absolute magnitude
coefficients = coefficients.sort_values(
    "abs_coefficient",
    ascending=False
).reset_index(drop=True)

print("==============================")
print("FEATURE COEFFICIENTS")
print("==============================")

print(
    coefficients[
        ["feature", "coefficient", "abs_coefficient"]
    ].to_string(index=False)
)

FEATURE COEFFICIENTS
              feature  coefficient  abs_coefficient
      idle_percentage    -9.861721         9.861721
           smoothness     9.281807         9.281807
          pause_count    -2.641517         2.641517
           mean_pause    -2.519736         2.519736
       total_distance     1.775870         1.775870
     velocity_entropy     1.539146         1.539146
          velocity_cv    -1.444474         1.444474
     std_acceleration     1.364807         1.364807
    mean_acceleration     1.174024         1.174024
    direction_entropy     0.273924         0.273924
        mean_velocity    -0.270513         0.270513
mean_direction_change     0.043717         0.043717
        mean_distance    -0.029182         0.029182
           num_events     0.000000         0.000000
